# VideoDB Understanding API Preview

This notebook walks through the full lifecycle for the analyzer-based Understanding API:

1. Connect to VideoDB
2. Pick a video
3. Create an Understanding workflow with named analyzers
4. List understandings for a video
5. Check workflow/analyzer status
6. Poll until completion using SDK-style `wait`, `poll_interval`, and `timeout` options
7. Fetch a specific analyzer output
8. Optionally delete the Understanding

> Analyzer inputs refer to upstream analyzers by **name**. The server generates internal analyzer IDs automatically.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Video-DB/videodb-cookbook/blob/preview/guides/preview/understanding.ipynb)


## 1. Install dependencies

Run this once per notebook environment.

In [ ]:
!pip install -q "git+https://github.com/Video-DB/videodb-python.git@feat/add-indexing-v2" python-dotenv


## 2. Connect to VideoDB

Set your API key in the environment as `VIDEO_DB_API_KEY`. If it is not set, the notebook will prompt for it.

In [ ]:
import os
import json
import time
from pprint import pprint
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()

os.environ["VIDEO_DB_API_KEY"] = "" 

conn = connect()

### Note about direct `conn.get()` / `conn.post()` paths

The HTTP endpoint is written as `/video/<video_id>/understand`, but when calling with the low-level VideoDB SDK connection, pass the path **without** the leading slash:

```python
conn.get(f"video/{video.id}/understand/{understanding_id}")
```

This avoids generating a URL with a double slash after the base URL.

## 3. Pick a collection and video

Set `COLLECTION_ID` and `VIDEO_ID` if you already know them. Otherwise, the helper cells below can show your collections/videos.

In [ ]:
# Optional: list collections
colls = conn.get_collections()
for i in colls:
    print(i)

In [ ]:
collection_id = "c-b42d21fd-6df6-46aa-bbdc-2518b7978876"
coll = conn.get_collection(collection_id)

In [ ]:
videos = coll.get_videos()
for i in videos:
    print(i)

In [ ]:
# Fill these in for your environment.
VIDEO_ID = "m-z-019edfb5-de31-7872-9c2d-3641bce53f90" 

video = coll.get_video(VIDEO_ID)

## 4. Create an Understanding

This example runs three analyzers:

- `speech`: transcribes the audio
- `objects`: detects objects in sampled frames
- `vlm_summary`: uses frames + speech + object detections to summarize the video

Notice that `vlm_summary.inputs` references `speech` and `objects` by **name**.

In [ ]:
understanding_payload = {
    "segmentation": {
        "type": "shot",
        "threshold": 5,
    },
    "sampling": {
        "strategy": "interval",
        "every": 1,
    },
    "analyzers": [
        {
            "name": "speech",
            "type": "speech_transcription",
            "config": {
                "model": "gemini-2.5-flash",
                "language": "en",
                "prompt": "Transcribe the spoken audio. Include timestamps when available and preserve important named entities.",
            },
        },
        {
            "name": "objects",
            "type": "object_detection",
            "sampling": {
                "strategy": "interval",
                "every": 1,
            },
            "config": {
                "labels": ["person", "vehicle", "animal", "product", "logo", "text", "scene_change"],
                "confidence_threshold": 0.35,
                "include_bounding_boxes": True,
            },
        },
        {
            "name": "vlm_summary",
            "type": "vlm",
            "inputs": ["speech", "objects"],
            "sampling": {
                "strategy": "uniform",
                "frame_count": 8,
            },
            "config": {
                "model": "gemini-2.5-flash",
                "prompt": (
                    "Using the sampled frames, object detections, and transcript, summarize what happens in the video. "
                    "Highlight key visual moments, objects, actions, on-screen text/logos, and how the narration/audio "
                    "relates to the visuals. Return concise JSON with fields: title, summary, timeline, "
                    "detected_objects, transcript_highlights, and confidence."
                ),
            },
        },
    ],
}

# Low-level SDK paths intentionally do not start with "/".
create_response = conn.post(
    f"video/{video.id}/understand",
    understanding_payload,
)
pprint(create_response)

understanding_id = create_response["understanding_id"]
print("Understanding ID:", understanding_id)

## 5. List understandings for a video

Use this to see all Understanding runs created for the video.

In [ ]:
understandings = conn.get(f"video/{video.id}/understand")
for i in understandings['understanding_results']:
    print("="*10)
    print(i['understanding_id'], i['status'])
    for i in i['analyzers']:
        print(i)

## 6. Get Understanding status

The status endpoint returns a minimal response: top-level status plus analyzer `id`, `name`, `type`, and `status`.

In [ ]:
understanding = conn.get(f"video/{video.id}/understand/{understanding_id}")


print("="*10, "Understanding", "="*10)
print(understanding['status'])

## 7. Get status for one analyzer

Pass `?analyzer=<name_or_id>` to filter the status response to one analyzer.

In [ ]:
speech_status = conn.get(
    f"video/{video.id}/understand/{understanding_id}",
    params={"analyzer": "speech"},
)
print(speech_status)

## 9. Fetch analyzer outputs

Fetch a specific analyzer's output by analyzer name or internal analyzer ID. The API hydrates the stored artifact and returns the output directly.

In [ ]:
speech_output = conn.get(
    f"video/{video.id}/understand/{understanding_id}/analyzers/speech/output"
)
pprint(speech_output)

In [ ]:
objects_output = conn.get(
    f"video/{video.id}/understand/{understanding_id}/analyzers/objects/output"
)
pprint(objects_output)

In [ ]:
vlm_output = conn.get(
    f"video/{video.id}/understand/{understanding_id}/analyzers/vlm_summary/output"
)
pprint(vlm_output)

## 10. Optional: delete the Understanding

Only run this if you want to remove the Understanding records.

In [ ]:
DELETE_UNDERSTANDING = False

if DELETE_UNDERSTANDING:
    delete_response = conn.delete(f"video/{video.id}/understand/{understanding_id}")
    pprint(delete_response)
else:
    print("Skipping delete. Set DELETE_UNDERSTANDING=True to delete this Understanding.")

## Endpoint summary

```text
POST   /video/<video_id>/understand
GET    /video/<video_id>/understand
GET    /video/<video_id>/understand/<understanding_id>
GET    /video/<video_id>/understand/<understanding_id>?analyzer=<name_or_id>
GET    /video/<video_id>/understand/<understanding_id>/analyzers/<name_or_id>/output
DELETE /video/<video_id>/understand/<understanding_id>
```

The internal callback endpoint is used by the Understanding service and is not normally called by users:

```text
POST /understanding/workflow_callback/<understanding_id>
```